# Memory Lab — why agents need memory, and how to build one

Chapter 2 gave the agent tools. This lab answers one question — **what can
tools not do?** — and then has you build the thing that can. It has two
independent halves:

| | What | You do |
|---|---|---|
| **Part A** | A five-minute A/B demo: the same agent with and without memory | Run it, read the two traces |
| **Part B** | The exercise: four TODOs driven over one long transcript | Implement `memory_starter.py` |

Every model call in this notebook is live (free GLM-4-Flash). Export your key
before starting Jupyter — **never paste a key into a cell**: the 6-point gate
item in Part B is this exact rule, applied to the agent.

```bash
export ZAI_API_KEY="..."   # then: jupyter lab
```

In [ ]:
# Setup — run this notebook from inside 03Memory/
import json, os, sys
sys.path.insert(0, os.path.abspath("."))

assert os.getenv("ZAI_API_KEY") or os.getenv("ZHIPU_API_KEY"), \
    "export ZAI_API_KEY before starting Jupyter (never put it in a cell)"

from tokens import estimator_name
from zhipu_client import ZhipuClient

client = ZhipuClient.from_env()
print("token estimator:", estimator_name())

---
## Part A · Why memory

A badge-audit assistant works across **two sessions**; each starts with a
fresh context.

**Session 1** states two facts, written to no file:
- the incident export is **`incident_0812.txt`**
- the fine is **200 yuan** per violating record

**Session 2** asks for the record count and the total fine. The answer is
split across the two places knowledge can live:

- `records = 7` exists **only inside the file** — only `read_file` reaches it
- *which* file, and the fine, were **only ever said** — the workspace holds
  three incident files (`0805` / `0812` / `0819`), so listing the directory
  does not disambiguate, and no file states the fine

First: chapter 2's agent, unchanged. Nothing survives session 1.

In [ ]:
from types import SimpleNamespace
import main
from zhipu_client import DEFAULT_MODEL

args = SimpleNamespace(model=DEFAULT_MODEL, implementation="solution",
                       session=None, max_steps=6, quiet=False)

tools_run = main.run_one("tools", args, main.make_client())

Read the session-2 trace above to the end. The tools are not broken — it
lists and reads files just fine. It fails because *which* incident and *what*
fine were never written anywhere its tools can reach, so it guesses a file
and invents a number (or gives up). That failure is structural.

Now the same agent, same tools, same loop — plus a memory pipeline that runs
when a session ends: extract → gate → reconcile → save.

In [ ]:
memory_run = main.run_one("memory", args, main.make_client())

In [ ]:
# The only thing that crossed the session boundary — a file you can read:
print(open("runs/memory.jsonl").read())

**That is the whole demo.** Two records, ~20 tokens, and session 2 starts
with them in a `[memory]` block. Tools extend what an agent can **do**;
memory extends what it can **keep**.

The demo is graded PASS/FAIL only. The machinery you just watched
(`write_memory`, `validate_record`, `reconcile`) is what you build in Part B.

---
## Part B · The four TODOs

No agent, no ReAct loop. Two inputs:

- **the seed store** — what "last month" left behind
- **the transcript** — one long handover conversation with everything planted:

| Planted | Exercises |
|---|---|
| new log path, "audit runs on day 1" | TODO 1 extraction → TODO 3 **ADD** |
| "the fine is **250** now, not 200" | TODO 3 **UPDATE** (old row → superseded) |
| "**stop sending** the report to security-team" | TODO 3 **DELETE** (revocation pass) |
| "the incident file is **still** incident_0812.txt" | TODO 3 **NOOP** |
| "keep the ops dashboard key ... `ZAI_API_KEY=sk-...`" said in the open | TODO 2 **secret** — extraction will surface it; the gate must stop it |
| a ~40-line config paste (same credential inside) | TODO 4 **L1 trim target** |
| "covers the ServerRoom **and** the Lab" | TODO 2 compound check |
| the assistant's own wrong arithmetic | TODO 1 must read **user turns only** |
| the transcript vs a 600-token budget | TODO 4 **L2 compaction** |

Look at the materials first:

In [ ]:
from sessions import (PIPELINE_BUDGET, PIPELINE_SYSTEM, SEED_MEMORY,
                      TRANSCRIPT, TRANSCRIPT_SESSION_NO)

print("seed store:")
for r in SEED_MEMORY:
    print(f"  {r['key']} = {r['value']}")
print(f"\ntranscript: {len(TRANSCRIPT)} messages; budget for STEP 4: {PIPELINE_BUDGET}t")
for m in TRANSCRIPT[:6]:
    print(f"  [{m['role']:<9}] {m['content'][:70]}...")

### Your task

Implement the four TODOs in **`memory_starter.py`** — the docstrings are the
full specification, and README §5 walks through each one (what the prompt
must say, the code steps, the toolbox). **The prompt placeholders at the top
of the file (EXTRACT / RECONCILE / REVOKE) are most of the real work** — the
method bodies are mostly plumbing. `trim_oversized` and `compact` are
provided. Stuck? `memory_starter_solution.py` is the standard answer:
consult after attempting, don't paste.

Before writing anything, look at your task list (free, runs on the mock):

In [ ]:
!{sys.executable} check.py

Expected on the untouched starter: **four red TODOs, all guardrails green**.
Red in YOUR TASK LIST means "not done yet"; red in GUARDRAILS means the
harness broke. The mock behind it is imperfect on purpose: its extractor
transcribes the credential, packs two facts into one record, and emits a
record with no value — the three candidates your gate must catch offline.

### Run your pipeline, step by step

**Testing happens on the real API**, one TODO at a time. In a terminal each
step is one command, saves its output under `runs/`
(`0_initial_memory.jsonl` → `1_todo1_candidates.json` → `2_todo2_gate.json`
→ `3_todo3_operations.json` → `final_memory.jsonl`), and ends with an
instant ✓/✗ feedback block:

```bash
python pipeline.py --implementation starter --step 1   # ... --step 2/3/4
```

The cells below run the same four steps inline so you can inspect every
intermediate as a live variable (nothing is written to `runs/` here). Flip
`USE_STARTER` to `True` to drive **your** implementation — fill the prompt
placeholders first.

In [ ]:
USE_STARTER = False   # True = your memory_starter.py, False = the reference

if USE_STARTER:
    from memory_starter import MemoryPolicy
else:
    from memory_agent import MemoryPolicy

from memory_store import MemoryStore

policy = MemoryPolicy()
store = MemoryStore()          # in-memory for the notebook
for r in SEED_MEMORY:
    store.add(dict(r))
store.operations.clear()
print("policy:", policy.name)

In [ ]:
# STEP 1 · TODO 1 — extract candidate records from the transcript (one API call)
candidates = policy.write_memory(client, TRANSCRIPT, TRANSCRIPT_SESSION_NO)
for c in candidates:
    print(" ·", json.dumps(c, ensure_ascii=False))

In [ ]:
# STEP 2 · TODO 2 — the write gate (no API call, on purpose)
accepted = []
for record in candidates:
    ok, reason = policy.validate_record(record, store)
    if ok:
        accepted.append(record)
        print(" PASS   ", json.dumps(record, ensure_ascii=False))
    else:
        store.log_rejection(record, reason)
        print(f" REJECT ({reason}):", json.dumps(record, ensure_ascii=False))

In [ ]:
# STEP 3 · TODO 3 — reconcile against the seed store (API x2: verdicts + revocations)
policy.reconcile(client, store, accepted, TRANSCRIPT, TRANSCRIPT_SESSION_NO)
print(store.report(TRANSCRIPT_SESSION_NO))
# The two rows worth staring at: ~ superseded (update keeps the audit trail)
# and x deleted (revocation marks, it does not erase).

In [ ]:
# STEP 4 · TODO 4 — assemble the transcript under the budget (the ladder)
history = [{"role": m["role"], "content": m["content"]} for m in TRANSCRIPT]
assembled = policy.build_context(client, PIPELINE_SYSTEM, store, history, PIPELINE_BUDGET)
print("\n".join(assembled.ladder))
print(f"-> {len(assembled.messages)} messages, {assembled.tokens:,} tokens")

In [ ]:
# The report card — 20 points, split by TODO (gate = 6, all or nothing)
from grader import grade_pipeline

grade = grade_pipeline(store, candidates, assembled, PIPELINE_BUDGET)
print("PASS" if grade.passed else "FAIL", f"— {grade.score}/{grade.total}")
for item in grade.feedback:
    print("  ·", item)

### The finish line

```bash
python pipeline.py --implementation starter  # full run: PASS — 20/20
grep "sk-" runs/final_memory.jsonl           # must print nothing
```

Run it two or three times — live models drift, and an occasional 19/20 on
the revocation item is variance, not a defect. For a deterministic deep
check, `python -m unittest discover -s tests` must show 0 failures, 0 skips.

Submit `memory_starter.py` + `runs/final_memory.jsonl`.

Optional closure: plug your pipeline back into Part A —
`python main.py --mode memory --implementation starter`.

---
### Discussion

1. Which of Part A's two failures is fixable with money — and what does that
   imply for system design?
2. Why may the write gate never call the model? Why does that make it the
   only fully unit-testable TODO?
3. Extraction reads user turns only. What real failure does that prevent?
4. The verifier requires the total to be an actual calculator Observation,
   but lets a *wrong* total through. Why is that boundary correct?